# Phase 2: Customer Segmentation (RFM + K-Means)

**Business question:** Who are our customers, and which groups deserve different
marketing treatment?

**Approach**
1. Build **Recency, Frequency, Monetary (RFM)** values for every real customer (`customer_unique_id`).
2. Score customers 1–5 on each dimension and map them to **rule-based segments** marketers can read at a glance.
3. Cluster customers with **K-Means** on log-scaled RFM features, choosing *k* with the elbow method and silhouette score.
4. Compare segments by share of customers vs. share of revenue, and turn that into actions.

All the logic lives in `src/segmentation.py`; this notebook walks through it step by step.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))  # notebook runs from notebooks/

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

import segmentation as seg
from db import get_engine

pd.set_option("display.float_format", "{:,.2f}".format)
engine = get_engine()

## 1. Build the RFM table
- **Recency:** days since the customer's last purchase, measured from one day after the last order in the data.
- **Frequency:** number of delivered orders.
- **Monetary:** lifetime spend (items + freight).

In [ ]:
rfm = seg.load_rfm(engine)
rfm[["recency_days", "frequency", "monetary"]].describe().T

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, col in zip(axes, ["recency_days", "frequency", "monetary"]):
    ax.hist(rfm[col], bins=50)
    ax.set_title(col)
axes[2].set_xscale("log")
plt.tight_layout()

repeat_rate = (rfm["frequency"] > 1).mean() * 100
print(f"Customers with more than one order: {repeat_rate:.1f}%")

**Observation:** Frequency barely varies. Most customers buy only once, so
frequency can't be split into quintiles like recency and monetary. It is scored
on fixed order counts instead (1 order = 1 ... 5+ orders = 5). Monetary is heavily
right-skewed, which is why the clustering step log-transforms it.

## 2. Rule-based RFM segments
R and M are scored by quintile (5 = best). Segment rules:

| Segment | Rule |
|---|---|
| Champions | 2+ orders and recent (R ≥ 4) |
| Loyal Customers | 2+ orders |
| New Big Spenders | Recent (R ≥ 4) and high spend (M ≥ 4) |
| Promising | Recent (R ≥ 4) |
| At Risk High Value | Not recent (R ≤ 2) and high spend (M ≥ 4) |
| Hibernating | Not recent (R ≤ 2) |
| Needs Attention | Everyone else |

In [ ]:
rfm = seg.score_rfm(rfm)
rfm_profile = seg.profile_segments(rfm, "rfm_segment")
rfm_profile[["customers", "customer_share_pct", "revenue_share_pct",
             "avg_recency_days", "avg_monetary", "avg_review_score"]]

## 3. K-Means clustering
Features are log-transformed (to tame the skew) and standardized (so recency in days
doesn't outweigh spend). *k* is chosen by the best silhouette score between 3 and 6,
a range a marketing team can actually act on.

In [ ]:
X = seg.build_features(rfm)
scores = seg.evaluate_k(X)
best_k = seg.choose_k(scores)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(scores.k, scores.inertia, marker="o"); a1.set(title="Elbow method", xlabel="k", ylabel="Inertia")
a2.plot(scores.k, scores.silhouette, marker="o", color="tab:orange")
a2.axvline(best_k, ls="--", color="grey"); a2.set(title=f"Silhouette (chosen k = {best_k})", xlabel="k")
plt.tight_layout()

In [ ]:
km = KMeans(n_clusters=best_k, n_init=10, random_state=seg.RANDOM_STATE).fit(X)
rfm["cluster"] = km.labels_
rfm["cluster_name"] = rfm["cluster"].map(seg.name_clusters(rfm, X))

cluster_profile = seg.profile_segments(rfm, "cluster_name")
cluster_profile[["customers", "customer_share_pct", "revenue_share_pct", "avg_recency_days",
                 "avg_frequency", "avg_monetary", "avg_review_score", "late_share"]]

In [ ]:
sample = rfm.sample(min(len(rfm), 5000), random_state=seg.RANDOM_STATE)
fig, ax = plt.subplots(figsize=(9, 6))
for name, g in sample.groupby("cluster_name"):
    ax.scatter(g["recency_days"], g["monetary"], s=8, alpha=0.5, label=name)
ax.set_yscale("log")
ax.set(title="Customer clusters: recency vs. lifetime spend",
       xlabel="Days since last purchase", ylabel="Lifetime spend (log scale)")
ax.legend(markerscale=3);

In [ ]:
p = cluster_profile.sort_values("revenue_share_pct")
y = np.arange(len(p))
fig, ax = plt.subplots(figsize=(9, 0.6 * len(p) + 1.5))
ax.barh(y - 0.2, p["customer_share_pct"], 0.4, label="% of customers")
ax.barh(y + 0.2, p["revenue_share_pct"], 0.4, label="% of revenue")
ax.set_yticks(y, p.index); ax.set_xlabel("Share (%)"); ax.legend()
ax.set_title("Which segments punch above their weight?");

## 4. How the two methods relate
Rows are K-Means clusters, columns are rule-based segments (% of each cluster).

In [ ]:
pd.crosstab(rfm["cluster_name"], rfm["rfm_segment"], normalize="index").mul(100).round(1)

## 5. Key findings and recommended actions
_Fill in from the tables above once you run this on the full dataset._

| Segment | Size | Revenue share | Recommended action |
|---|---|---|---|
| Loyal Repeat Buyers | __% | __% | VIP perks and early access; protect at all costs |
| At-Risk / High Spenders (lapsed) | __% | __% | Win-back campaign with a personalized discount |
| Recent Buyers | __% | __% | Second-purchase nudge within 30 days |
| Low Spenders | __% | __% | Low-cost automated email only |

**Headline insight to write up:** _e.g. "Only __% of customers ever buy twice, but they
generate __% of revenue: converting first-time buyers into repeat buyers is the biggest
growth lever, which motivates the churn model in Phase 3."_

## 6. Save results
Writes the `customer_segments` table (used by the churn model and dashboard) and
the charts in `outputs/figures/`.

In [ ]:
seg.run(k=best_k);